# Flow viz — hub Sankey, product detail, zone map

Lightweight drill-down notebook. Change **one config cell** and re-run the
viz cells below.

- **Notebook 04** = batch pipeline (roles, patterns, CSV exports)
- **This notebook** = storytelling / teammate discussions
- Logic lives in `viz_flows.py`

**Grains:** `installation` | `complex` | `zone`

**Cargo grain:** `product` | `group` | `family` — set `CARGO_GRAIN` in config. Flows
still use a `Product` column, but values roll up to Kpler Group or Family when
selected.

**Flow scope:** `all_spain_touched` (default) = any edge with Spain on origin
or destination. Loads a grain-specific cache CSV if present (e.g.
`owner_od_flows_all_spain_touched_group.csv`), else queries DuckDB.

In [103]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import viz_flows
importlib.reload(viz_flows)  # pick up viz_flows.py edits without kernel restart

from viz_flows import (
    CARGO_LABEL,
    DEFAULT_OD_CARGO,
    aggregate_zone_edges,
    build_hub_trade_edges,
    hub_dashboard,
    load_entity_map,
    load_od_flows,
    load_zone_coords,
    make_flow_dashboard,
    plot_hub_trade_bars,
    plot_hub_trade_sankey,
    plot_product_flow_sankey,
    plot_zone_flow_map,
    resolve_entity,
    resolve_od_cargo,
)

DERIVED = ROOT / "data" / "derived"
ENTITY_MAP_PATH = DERIVED / "terminal_entity_map.csv"
ZONE_COORDS_PATH = DERIVED / "zone_coords.csv"

# --- config (edit here) ---
FOCUS_COUNTRY = "Spain"
FLOW_SCOPE = "all_spain_touched"  # domestic | all_spain_touched | cross_border

# Hub drill-down
ENTITY = "APM Terminals Barcelona"
ENTITY_GRAIN = "installation"#"zone"  # installation | complex | zone
COUNTERPARTY_GRAIN = "zone"  # roll partners to this grain in hub Sankey

# Cargo grain: product | group | family (Kpler hierarchy; flows use Product column)
CARGO_GRAIN = "product"
OD_CARGO = None  # None → DEFAULT_OD_CARGO[CARGO_GRAIN]

# Detail panel — set a single cargo value at the chosen grain, or None for hub Sankey
PRODUCT = None  # e.g. "Gasoil" (product) or "Gasoil/Diesel" (group)

TOP_N = 12
MIN_EDGE_KB = 1.0
MAP_TOP_EDGES = 20
LAYOUT = "separate"  # separate keeps legends; dual merges panels without legends

OD_FILTER = resolve_od_cargo(OD_CARGO, cargo_grain=CARGO_GRAIN)

print("ROOT", ROOT)
print("viz_flows", viz_flows.__file__)
print(
    "hub:", ENTITY, "@", ENTITY_GRAIN,
    "| cargo:", CARGO_LABEL[CARGO_GRAIN],
    "| filter:", OD_FILTER,
    "| detail:", PRODUCT,
)

ROOT C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\prod_nowcast
viz_flows C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\prod_nowcast\viz_flows.py
hub: APM Terminals Barcelona @ installation | cargo: Product | filter: ['Naphtha', 'Gasoil', 'Gasoline', 'Jet', 'FO'] | detail: None


In [104]:
entity_map = load_entity_map(ENTITY_MAP_PATH)
zone_coords = load_zone_coords(ZONE_COORDS_PATH)
hub = resolve_entity(ENTITY, ENTITY_GRAIN, entity_map)

flows, flow_source = load_od_flows(
    focus_country=FOCUS_COUNTRY,
    scope=FLOW_SCOPE,
    derived_dir=DERIVED,
    products=OD_FILTER,
    cargo_grain=CARGO_GRAIN,
)

# Uncomment after first DuckDB load to cache for faster re-runs:
# from viz_flows import cache_od_flows
# print("cached →", cache_od_flows(flows, derived_dir=DERIVED, scope=FLOW_SCOPE, cargo_grain=CARGO_GRAIN))

print(f"flows: {len(flows):,} edges | source={flow_source} | kb={flows['kb'].sum():,.0f}")
print(f"hub resolved: {hub.label!r} grain={hub.grain} zone={hub.zone!r}")
flows.head(3)

flows: 5,528 edges | source=duckdb | kb=2,418,795
hub resolved: 'APM Terminals Barcelona' grain=installation zone='Barcelona'


,origin_installation,dest_installation,origin_country,dest_country,origin_zone,dest_zone,Product,seller,buyer,n_trades,...,seller_eq_buyer,flow_class,origin_role,origin_import_share,origin_net_kb,dest_role,dest_import_share,dest_net_kb,surplus_to_deficit,cargo_grain
0,Torre Arenillas,Evos Amsterdam East,Spain,Netherlands,Huelva,Amsterdam Port,Naphtha,Moeve,(unknown),7,...,False,commercial_to_3rd,supply_center,0.025754,-70158.178,demand_center,0.819370,46039.594,True,product
1,Repsol Coruna Refinery,Mohammedia Refinery,Spain,Morocco,Coruna,Mohammedia,Gasoline,Repsol,SAMIR,1,...,False,commercial_to_3rd,supply_center,0.050895,-31186.170,demand_center,0.975796,23323.140,True,product
2,Repsol Cartagena Refinery,Versalis Cracker Brindisi,Spain,Italy,Cartagena,Brindisi,Naphtha,Repsol,Eni,17,...,False,commercial_to_3rd,supply_center,0.022735,-140821.216,demand_center,0.994404,30084.268,True,product


## Hub dashboard (map + detail)

Use **`LAYOUT = "separate"`** (recommended): proper geo map + Sankey with legends.

`dual` merges both panels in one figure but **drops Sankey/map legends** — only use
for quick previews.

In [105]:
result = hub_dashboard(
    flows,
    hub,
    entity_map=entity_map,
    zone_coords=zone_coords,
    focus_country=FOCUS_COUNTRY,
    product=PRODUCT,
    counterparty_grain=COUNTERPARTY_GRAIN,
    top_n=TOP_N,
    min_kb=MIN_EDGE_KB,
    map_top_edges=MAP_TOP_EDGES,
    layout=LAYOUT,
    products=OD_FILTER,
    cargo_grain=CARGO_GRAIN,
)

if LAYOUT == "separate":
    fig_map, fig_detail = result
    fig_map.show()
    fig_detail.show()
else:
    result.show()

## Hub trade Sankey + tornado bars

Standalone hub view (no map). Works for **installation**, **complex**, or **zone**.

In [106]:
hub_in = build_hub_trade_edges(
    flows,
    hub,
    direction="inbound",
    counterparty_grain=COUNTERPARTY_GRAIN,
    entity_map=entity_map,
    products=OD_FILTER,
    top_n=TOP_N,
    min_kb=MIN_EDGE_KB,
)
hub_out = build_hub_trade_edges(
    flows,
    hub,
    direction="outbound",
    counterparty_grain=COUNTERPARTY_GRAIN,
    entity_map=entity_map,
    products=OD_FILTER,
    top_n=TOP_N,
    min_kb=MIN_EDGE_KB,
)

display(
    pd.concat(
        [hub_in.assign(direction="inbound"), hub_out.assign(direction="outbound")],
        ignore_index=True,
    ).sort_values("kb", ascending=False)
)

fig_hub = plot_hub_trade_sankey(
    hub.label,
    hub_in,
    hub_out,
    title=(
        f"Hub trade — {hub.label} ({hub.grain}; partners @ {COUNTERPARTY_GRAIN})"
    ),
    cargo_grain=CARGO_GRAIN,
)
fig_hub.show()

fig_bars = plot_hub_trade_bars(
    hub_in,
    hub_out,
    title=f"Hub profile — {hub.label}",
    cargo_grain=CARGO_GRAIN,
)
fig_bars.show()

,counterparty,Product,kb,n_trades,direction
0,Sikka,Jet,5405.391,18,inbound
1,Corinth,Jet,4787.077,19,inbound
2,Tarragona,FO,3320.899,35,inbound
3,Mina Al-Ahmadi,Jet,2129.267,9,inbound
4,Sarroch,Gasoil,2088.255,10,inbound
5,Augusta,Gasoil,1710.301,8,inbound
6,Ruwais,Jet,1686.880,6,inbound
7,Jubail Industrial Port,Jet,1668.585,6,inbound
8,Aliaga,Jet,1452.850,6,inbound
9,Cartagena,Jet,1237.466,10,inbound


## Product detail panel

Top zone→zone (or installation) edges for one product, optionally scoped to hub.

In [107]:
if PRODUCT:
    fig_prod = plot_product_flow_sankey(
        flows,
        product=PRODUCT,
        hub=hub,
        display_grain="zone",
        top_n=MAP_TOP_EDGES,
        min_kb=MIN_EDGE_KB,
    )
    fig_prod.show()
else:
    print("Set PRODUCT in config to render product detail Sankey.")

Set PRODUCT in config to render product detail Sankey.


## Country zone flow map

All top zone arcs touching the focus country (not hub-filtered).

In [108]:
country_zone_edges = aggregate_zone_edges(
    flows,
    focus_country=FOCUS_COUNTRY,
    product=PRODUCT,
    min_kb=MIN_EDGE_KB,
    top_n=MAP_TOP_EDGES,
)

highlight = {hub.zone} if hub.zone else set()
fig_country_map = plot_zone_flow_map(
    country_zone_edges,
    zone_coords,
    focus_country=FOCUS_COUNTRY,
    title=(
        f"{FOCUS_COUNTRY} zone flows"
        + (f" · {PRODUCT}" if PRODUCT else "")
        + f" (top {MAP_TOP_EDGES})"
    ),
    highlight_zones=highlight,
    cargo_grain=CARGO_GRAIN,
)
fig_country_map.show()
country_zone_edges

,origin_zone,dest_zone,origin_country,dest_country,Product,kb,n_trades
0,Cartagena,Aratu,Spain,Brazil,Naphtha,31209.048,99
1,Rotterdam,Algeciras,Netherlands,Spain,FO,25394.915,114
2,Las Palmas,Conakry,Spain,Guinea,Gasoil,24582.305,132
3,Cartagena,Tarragona,Spain,Spain,Naphtha,23966.123,139
4,Las Palmas,Nouadhibou,Spain,Mauritania,Gasoil,21405.227,209
5,Huelva,Tenerife,Spain,Spain,Gasoil,20056.810,205
6,Huelva,New York,Spain,United States,Naphtha,19683.839,117
7,Algeciras,Las Palmas,Spain,Spain,Gasoil,18697.248,153
8,Donges,Algeciras,France,Spain,FO,17296.303,82
9,Sines,Las Palmas,Portugal,Spain,FO,17128.381,95


## Quick reference — example configs

| Goal | ENTITY | ENTITY_GRAIN | CARGO_GRAIN | PRODUCT |
|---|---|---|---|---|
| Cartagena island supply | `Cartagena` | `zone` | `product` | `Gasoil` |
| Middle distillates rollup | `Barcelona` | `zone` | `group` | `Gasoil/Diesel` |
| Barcelona port complex | `Barcelona Port` | `complex` | `product` | `Diesel` |
| Single berth | `Barcelona 32C` | `installation` | `product` | `Gasoil` |
| La Rabida system | `La Rabida refinery & port` | `complex` | `product` | `Naphtha` |
| Algeciras bunkering | `Algeciras Storage` | `complex` | `product` | `FO` |